# CA21 Demo

This notebook demonstrates a minimal debug run for the CA21 assignment. It is intentionally lightweight and designed to run quickly using the `configs/debug.yaml` settings.


In [ ]:
from pathlib import Path
import importlib.util
import torch
import random
import numpy as np

# load modules directly from src (avoids package name issues in class assignments)
base = Path(__file__).resolve().parents[2] / "src"
spec = importlib.util.spec_from_file_location("ca21.model", str(base / "model.py"))
model_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_mod)  # type: ignore

# utils for seeding
spec_utils = importlib.util.spec_from_file_location("ca21.utils", str(base / "utils.py"))
utils_mod = importlib.util.module_from_spec(spec_utils)
spec_utils.loader.exec_module(utils_mod)  # type: ignore

Policy = model_mod.MLPPolicy
Value = model_mod.MLPValue

# seed and device detection (notebook should be reproducible and quick with debug config)
utils_mod.set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = dict(input_dim=8, hidden_dim=32, action_dim=4, device=str(device))
policy = Policy(**cfg).to(device)
value = Value(input_dim=cfg["input_dim"], hidden_dim=cfg["hidden_dim"]).to(device)

x = torch.randn(4, cfg["input_dim"]).to(device)
print("logits shape:", policy(x).shape)
print("value shape:", value(x).shape)
print("device:", device)